In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
# Preparation: using data from 8c1834d commit:

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
# Q1: How many lesson pages are in the dataset?

len(documents)

# 72

72

In [6]:
"""
Q2: Index the documents with minsearch - make content a text field and filename a keyword field. 
Then search with this query:

How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?
"""

from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [7]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [8]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={'content': 2.0},
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [9]:
[doc["filename"] for doc in search_results]

['01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/11-agents-intro.md',
 '01-agentic-rag/lessons/16-other-frameworks.md']

In [10]:
# Q3: Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?
# How does the agentic loop keep calling the model until it stops?


from rag_helper import RAGBase

# RAGBase was written for the FAQ schema (section/question/answer), while our documents have filename and content.
# Taking RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context

rag = RAGBase(index=index, llm_client=openai_client)

def custom_search(self, query, num_results=5):
        boost_dict = {'content': 3.0, 'section': 0.5}
        filter_dict = {'filename': self.filename}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

def custom_build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append('content: ' + doc['content'])
            lines.append('filename: ' + doc['filename'])
            lines.append('')

        return '\n'.join(lines).strip()


# Overriding the two methods:

import types
rag.search = types.MethodType(custom_search, rag)
rag.build_context = types.MethodType(custom_build_context, rag)

In [11]:
question = "How does the agentic loop keep calling the model until it stops?"

prompt = rag.build_prompt(question, search_results)
prompt

'QUESTION: How does the agentic loop keep calling the model until it stops?\n\nCONTEXT:\ncontent: # The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions

In [12]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt
)

In [13]:
response.output_text

'It keeps going with a `while True` loop plus a flag.\n\nHow it works:\n\n1. Send the current `messages` to the model.\n2. Read the model output.\n3. If the output contains any `function_call` items:\n   - run the tool(s),\n   - append the tool results to `messages`,\n   - set `has_function_calls = True`.\n4. If there are no function calls, the model has given a final answer, so break the loop.\n\nSo the stopping condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn other words, the agent doesn’t decide “I’m done” in your code. The model decides by either:\n- asking for more tool calls, which makes the loop continue, or\n- returning only a normal message, which makes the loop stop.\n\nA simplified view:\n\n```python\nwhile True:\n    response = model(messages)\n\n    if response has tool calls:\n        run tools\n        add tool outputs to messages\n    else:\n        break\n```\n\nThe key idea is that the model is “driving,” and your loop just keeps repla

In [14]:
response.usage

ResponseUsage(input_tokens=7083, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=246, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7329)

In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [16]:
# Q4: How many chunks do you get?

print(len(chunks))

295


In [17]:
# Q5: RAG with chunking. Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?


index_chuncks = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index_chuncks.fit(chunks)

In [18]:
rag_chunck = RAGBase(index=index_chuncks, llm_client=openai_client)

def custom_search(self, query, num_results=5):
        boost_dict = {'content': 3.0}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict
        )

def custom_build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append('content: ' + doc['content'])
            lines.append('filename: ' + doc['filename'])
            lines.append('')

        return '\n'.join(lines).strip()


# Overriding the two methods:

import types
rag_chunck.search = types.MethodType(custom_search, rag_chunck)
rag_chunck.build_context = types.MethodType(custom_build_context, rag_chunck)

In [19]:
search_results_chunck = rag_chunck.search(question)
prompt_chunck = rag_chunck.build_prompt(question, search_results_chunck)
prompt_chunck

'QUESTION: How does the agentic loop keep calling the model until it stops?\n\nCONTEXT:\ncontent: while` loop. The loop keeps calling the model until\nit returns a response without any function calls. We also keep an\niteration counter so we can see how many round-trips happened.\n\n```python\nit = 1\n\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            print("function_call:", item.name, item.arguments)\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n\n        elif item.type == "message":\n            print("ASSISTANT:")\n            print(item.content[0].text)\n\n    it = it + 1\n    if has_function_calls == False

In [20]:
response_chunck = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt_chunck
)

In [21]:
response_chunck.output_text

'The agentic loop keeps calling the model with a `while True` loop.\n\nHow it stops:\n\n1. Call the model.\n2. Check the model’s output.\n3. If it includes any `function_call` items, run those tools and set `has_function_calls = True`.\n4. Append the tool results to the conversation.\n5. Loop again, so the model can see the tool output.\n6. If there are no function calls in that turn, break out of the loop.\n\nSo the stopping condition is simply:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nThat means the model has returned a final answer instead of asking for more tools.\n\nIn short: the model decides whether more tool use is needed, and the loop keeps going until the model stops requesting functions.'

In [22]:
response_chunck.usage

ResponseUsage(input_tokens=2266, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=167, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2433)

In [ ]:
# Q6. Turning it into an agent. How many times did the agent call search?

import json
from openai import OpenAI

client = OpenAI()

def search(query: str) -> str:
    """Search the course content index for relevant chunks matching the query."""
    results = index_chuncks.search(query, num_results=5)
    return "\n\n".join(r["content"] for r in results)

tool_schema = [{
    "type": "function",
    "name": "search",
    "description": "Search the course content index for relevant chunks matching the query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query"}
        },
        "required": ["query"]
    }
}]

question = "How does the agentic loop work, and how is it different from plain RAG?"
messages = [
    {"role": "developer", "content": "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."},
    {"role": "user", "content": question}
]

search_count = 0

while True:
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=tool_schema
    )
    
    tool_calls = [item for item in response.output if item.type == "function_call"]
    
    if not tool_calls:
        final = [item for item in response.output if item.type == "message"]
        if final:
            print(final[0].content[0].text)
        break
    
    for tc in tool_calls:
        args = json.loads(tc.arguments)
        result = search(args["query"])
        search_count += 1
        messages.append(tc)
        messages.append({"type": "function_call_output", "call_id": tc.call_id, "output": result})

print(f"\nSearch was called {search_count} times")

In [52]:
import toyaikit
print(dir(toyaikit))

['AnthropicClient', 'AnthropicMessagesRunner', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'chat', 'llm', 'pricing', 'tools']


In [54]:
print(dir(llm))


['AnthropicClient', 'BaseModel', 'ChatCompletion', 'LLMClient', 'List', 'Message', 'OpenAI', 'OpenAIChatCompletionsClient', 'OpenAIClient', 'Optional', 'ParsedChatCompletion', 'ParsedResponse', 'RawMessageStopEvent', 'Response', 'Tools', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']


In [58]:
import inspect
from toyaikit.tools import Tools
print(inspect.getsource(Tools.__init__))

    def __init__(self):
        self.tools = {}
        self.functions = {}

